In [0]:
# ClinicalTrials.gov API에 직접 HTTP 요청을 보내
# 데이터 수집 과정을 수동 다운로드에서 코드 기반 ingestion으로 전환한다.

import requests
import json
import time

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

PAGE_SIZE = 10
MAX_PAGES = 10

In [0]:
params = {
    "pageSize": PAGE_SIZE
}

response = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

response.raise_for_status()

data = response.json()

print("status:", response.status_code)
print("studies:", len(data["studies"]))
print("has next token:", "nextPageToken" in data)

next_page_token = data.get("nextPageToken")

print(next_page_token)

status: 200
studies: 10
has next token: True
ZVNj7o2Elu8o3lpwW8iooeytmo6Qfp5nY_Gm2fg


In [0]:
params = {
    "pageSize": PAGE_SIZE,
    "pageToken": next_page_token
}

response2 = requests.get(
    BASE_URL,
    params=params,
    timeout=30
)

response2.raise_for_status()

data2 = response2.json()

print(len(data2["studies"]))
print(data2.get("nextPageToken"))

10
ZVNj7o2Elu8o3lpwW8iooeytmo6QfZ5qZPOm2fg


In [0]:
all_studies = []

next_page_token = None

for page in range(MAX_PAGES):

    params = {
        "pageSize": PAGE_SIZE
    }

    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    studies = data.get("studies", [])

    all_studies.extend(studies)

    print(
        f"page={page + 1}, "
        f"received={len(studies)}, "
        f"total={len(all_studies)}"
    )

    next_page_token = data.get("nextPageToken")

    if not next_page_token:
        print("No more pages.")
        break

    time.sleep(0.2)

page=1, received=10, total=10
page=2, received=10, total=20
page=3, received=10, total=30
page=4, received=10, total=40
page=5, received=10, total=50
page=6, received=10, total=60
page=7, received=10, total=70
page=8, received=10, total=80
page=9, received=10, total=90
page=10, received=10, total=100


In [0]:
%sql
-- # ClinicalTrials.gov API의 원본 응답을 변형 없이 보존하기 위한 Raw Volume을 생성한다.
-- # Bronze Table과 달리 API 응답 파일 자체를 저장해 재처리 및 원본 추적이 가능하도록 한다.
CREATE VOLUME IF NOT EXISTS clinical_trials.raw_api;

In [0]:
# 전체 데이터를 Python 메모리에 누적하지 않고 페이지 단위로 즉시 저장한다.
# 이를 통해 데이터 규모 증가에 따른 메모리 사용을 줄이고,
# 수집 중 장애가 발생하더라도 이미 수집한 Raw 데이터를 보존한다.

import requests
import json
import time
from datetime import date
import os

catalog = spark.sql(
    "SELECT current_catalog()"
).first()[0]

ingest_date = date.today().isoformat()

raw_dir = (
    f"/Volumes/{catalog}/clinical_trials/raw_api/"
    f"ingest_date={ingest_date}"
)

os.makedirs(raw_dir, exist_ok=True)

print(raw_dir)

# 언제 수집한 데이터인지 물리적인 저장 구조에서도 알 수 있게 하기 위해서

BASE_URL = "https://clinicaltrials.gov/api/v2/studies"

PAGE_SIZE = 1000
MAX_PAGES = 10

next_page_token = None
total_studies = 0

for page_num in range(1, MAX_PAGES + 1):

    params = {
        "pageSize": PAGE_SIZE
    }

    if next_page_token:
        params["pageToken"] = next_page_token

    response = requests.get(
        BASE_URL,
        params=params,
        timeout=30
    )

    response.raise_for_status()

    data = response.json()

    studies = data.get("studies", [])

    # 이번 페이지의 원본 API 응답을 즉시 저장
    file_path = (
        f"{raw_dir}/page_{page_num:04d}.json"
    )

    with open(
        file_path,
        "w",
        encoding="utf-8"
    ) as f:
        json.dump(
            data,
            f,
            ensure_ascii=False
        )

    total_studies += len(studies)

    print(
        f"page={page_num}, "
        f"received={len(studies)}, "
        f"total={total_studies}, "
        f"saved={file_path}"
    )

    next_page_token = data.get("nextPageToken")

    if not next_page_token:
        print("No more pages.")
        break

    time.sleep(0.2)

/Volumes/workspace/clinical_trials/raw_api/ingest_date=2026-09-25
page=1, received=1000, total=1000, saved=/Volumes/workspace/clinical_trials/raw_api/ingest_date=2026-09-25/page_0001.json
page=2, received=1000, total=2000, saved=/Volumes/workspace/clinical_trials/raw_api/ingest_date=2026-09-25/page_0002.json
page=3, received=1000, total=3000, saved=/Volumes/workspace/clinical_trials/raw_api/ingest_date=2026-09-25/page_0003.json
page=4, received=1000, total=4000, saved=/Volumes/workspace/clinical_trials/raw_api/ingest_date=2026-09-25/page_0004.json
page=5, received=1000, total=5000, saved=/Volumes/workspace/clinical_trials/raw_api/ingest_date=2026-09-25/page_0005.json
page=6, received=1000, total=6000, saved=/Volumes/workspace/clinical_trials/raw_api/ingest_date=2026-09-25/page_0006.json
page=7, received=1000, total=7000, saved=/Volumes/workspace/clinical_trials/raw_api/ingest_date=2026-09-25/page_0007.json
page=8, received=1000, total=8000, saved=/Volumes/workspace/clinical_trials/raw_

In [0]:
# 실제 파일이 생겼는지 확인합니다

files = sorted(os.listdir(raw_dir))

print("file count:", len(files))

for file in files[:10]:
    print(file)

file count: 10
page_0001.json
page_0002.json
page_0003.json
page_0004.json
page_0005.json
page_0006.json
page_0007.json
page_0008.json
page_0009.json
page_0010.json


In [0]:
raw_pages_df = (
    spark.read
    .option("multiLine", True)
    .json(f"{raw_dir}/page_*.json")
)

print("Raw API pages:", raw_pages_df.count())
# 1 file = 1 Spark row

Raw API pages: 10


In [0]:
# Study 단위로 explode
from pyspark.sql import functions as F

studies_10k_df = (
    raw_pages_df
    .select(
        F.explode("studies").alias("study")
    )
)

print(studies_10k_df.count())

10000


In [0]:
# NCT ID 중복 확인

duplicates_df = (
    studies_10k_df
    .select(
        F.col(
            "study.protocolSection.identificationModule.nctId"
        ).alias("nct_id")
    )
    .groupBy("nct_id")
    .count()
    .filter(F.col("count") > 1)
)

print(
    "duplicate NCT IDs:",
    duplicates_df.count()
)

duplicate NCT IDs: 0


In [0]:
# Bronze Table 교체

# Delta Table은 기존 테이블 schema와 write 대상 DataFrame의 schema가
# 다를 경우 기본적으로 write를 거부한다.
# 이는 의도하지 않은 schema 변경으로 기존 데이터 구조가 깨지는 것을 방지한다. Cell 13 참조
#  .option("overwriteSchema", "true")

bronze_10k_df = (
    studies_10k_df
    .withColumn(
        "source",
        F.lit("clinicaltrials.gov")
    )
    .withColumn(
        "ingested_at",
        F.current_timestamp()
    )
    .withColumn(
        "ingest_date",
        F.lit(ingest_date).cast("date")
    )
)

(
    bronze_10k_df.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "clinical_trials.bronze_studies"
    )
)

spark.read.table(
    "clinical_trials.bronze_studies"
).count()

10000

In [0]:
%sql
-- Table History
DESCRIBE HISTORY clinical_trials.bronze_studies;

version,timestamp,userId,userName,operation,operationParameters,job,notebook,queryHistoryStatementId,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
1,2026-09-25T16:10:15.000Z,71077798721055,warwarsn@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, canOverwriteSchema -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(363666689905209),b8d182b1-49f2-425b-b629-f9b04081420c,0925-160029-pga7lm0n-v2n,0,WriteSerializable,false,"Map(numFiles -> 4, numRemovedFiles -> 1, numRemovedBytes -> 3363299, numDeletionVectorsRemoved -> 0, numOutputRows -> 10000, numOutputBytes -> 31897416)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13
0,2026-09-25T08:26:57.000Z,71077798721055,warwarsn@gmail.com,CREATE OR REPLACE TABLE AS SELECT,"Map(isV1SaveAsTableOverwrite -> true, partitionBy -> [], clusterBy -> [], description -> null, isManaged -> true, properties -> {""delta.parquet.format.version"":""2.12.0"",""delta.parquet.format.version.afe.internal"":""2.12.0"",""delta.parquet.compression.codec"":""zstd"",""delta.enableDeletionVectors"":""true""}, statsOnLoad -> true)",null,List(26051292554930),fe5f2532-449b-4dc0-a586-550e4c7c8e02,0925-082200-iij2g1fz-v2n,null,WriteSerializable,false,"Map(numFiles -> 1, numRemovedFiles -> 0, numRemovedBytes -> 0, numDeletionVectorsRemoved -> 0, numOutputRows -> 1000, numOutputBytes -> 3363299)",null,Databricks-Runtime/19.6.x-aarch64-photon-scala2.13


In [0]:
existing_bronze_df = spark.read.table(
    "clinical_trials.bronze_studies"
)

print("Existing columns:")
print(existing_bronze_df.columns)

print("New columns:")
print(bronze_10k_df.columns)

Existing columns:
['study', 'source', 'ingested_at', 'source_file']
New columns:
['study', 'source', 'ingested_at', 'ingest_date']
